<a href="https://colab.research.google.com/github/KeshavFTW099/ArtificalAdvanced_NeuralNetworks/blob/main/099_WeekANN_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow.keras.datasets as datasets
import tensorflow.keras.layers as layers
import tensorflow.keras.models as kmodels
import tensorflow.keras.callbacks
import tensorflow.keras.preprocessing as image
import tensorflow.keras.preprocessing.image as ImageGenerator
from tensorflow.keras.utils import to_categorical,plot_model
import numpy as np
import pandas as pd

In [ ]:
(x_train,y_train),(x_test,y_test)=datasets.cifar100.load_data()

169001437/169001437 ━━━━━━━━━━━━━━━━━━━━ 909s 5us/step


In [ ]:
x_train=x_train/255
x_test=x_test/255

In [ ]:
y1_train=to_categorical(y_train,100)
y1_test=to_categorical(y_test,100)

In [ ]:
traindata_generator=ImageGenerator.ImageDataGenerator(rotation_range=10,
                                   zoom_range=0.1,
                                   width_shift_range=0.1,
                                   height_shift_range=0.1,
                                   shear_range=0.1,
                                   horizontal_flip=True)
validation_generator=ImageGenerator.ImageDataGenerator(rescale=1./255)
testdata_generator=ImageGenerator.ImageDataGenerator(rescale=1./255)

In [ ]:
batch_size=32
epochs=20

In [ ]:
def inceptionnet(x,filters):
  # Branch 1: 1x1 convolution
  layer1=layers.Conv2D(filters=filters[0],kernel_size=(1,1),strides=(1,1),padding='same',activation='relu')(x)

  # Branch 2: 1x1 convolution followed by 3x3 convolution
  path2_reduce=layers.Conv2D(filters=filters[1][0],kernel_size=(1,1),strides=(1,1),padding='same',activation='relu')(x)
  layer2=layers.Conv2D(filters=filters[1][1],kernel_size=(3,3),strides=(1,1),padding='same',activation='relu')(path2_reduce)

  # Branch 3: 1x1 convolution followed by 5x5 convolution
  path3_reduce=layers.Conv2D(filters=filters[2][0],kernel_size=(1,1),strides=(1,1),padding='same',activation='relu')(x)
  layer3=layers.Conv2D(filters=filters[2][1],kernel_size=(5,5),strides=(1,1),padding='same',activation='relu')(path3_reduce)

  # Branch 4: 3x3 max pooling followed by 1x1 convolution
  path4_pool=layers.MaxPooling2D(pool_size=(3,3),strides=(1,1),padding='same')(x)
  layer4=layers.Conv2D(filters=filters[3],kernel_size=(1,1),strides=(1,1),padding='same',activation='relu')(path4_pool)

  return layers.concatenate([layer1, layer2, layer3, layer4], axis=-1)

In [ ]:
def helperfunction(x, name=None):
    layer=layers.AveragePooling2D(pool_size=(3,3), strides=3, padding="valid")(x)
    layer=layers.Conv2D(filters=128, kernel_size=(1,1), strides=1, padding="same", activation="relu")(layer)
    layer=layers.Flatten()(layer)
    layer=layers.Dense(units=255, activation="relu")(layer)
    layer=layers.Dropout(0.4)(layer)
    layer=layers.Dense(units=100, activation="softmax", name=name)(layer)
    return layer

In [ ]:
def googlenet():
    inputlayer=layers.Input(shape=(32,32,3))

    layer=layers.Conv2D(filters=64, kernel_size=(7,7), strides=1, padding="same", activation="relu")(inputlayer)
    layer=layers.MaxPooling2D(pool_size=(3,3), strides=2, padding="same")(layer)
    layer=layers.BatchNormalization()(layer)

    layer=layers.Conv2D(filters=64, kernel_size=(1,1), strides=1, padding="same", activation="relu")(layer)
    layer=layers.Conv2D(filters=192, kernel_size=(3,3), strides=1, padding="same", activation="relu")(layer)
    layer=layers.BatchNormalization()(layer)
    layer=layers.MaxPooling2D(pool_size=(3,3), strides=2, padding="same")(layer)

    layer=inceptionnet(layer, [64, (96, 128), (16, 32), 32])
    layer=inceptionnet(layer, [128, (128, 192), (32, 96), 64])
    layer=layers.MaxPooling2D(pool_size=(3,3), strides=2, padding="same")(layer)

    layer=inceptionnet(layer, [192, (96, 208), (16, 48), 64])
    final_0=helperfunction(layer, name="final_layer_0")
    layer=inceptionnet(layer, [160, (112, 224), (24, 64), 64])
    layer=inceptionnet(layer, [128, (128, 256), (24, 64), 64])
    layer=inceptionnet(layer, [112, (144, 288), (32, 64), 64])
    final_1=helperfunction(layer, name="final_layer_1")
    layer=inceptionnet(layer, [256, (160, 320), (32, 128), 128])
    layer=layers.MaxPooling2D(pool_size=(3,3), strides=2, padding="same")(layer)

    layer=inceptionnet(layer, [256, (160, 320), (32, 128), 128])
    layer=inceptionnet(layer, [384, (192, 384), (48, 128), 128])
    layer=layers.AveragePooling2D(pool_size=(7,7), strides=2, padding="same")(layer)

    layer=layers.Flatten()(layer)
    layer=layers.Dropout(0.4)(layer)
    layer=layers.Dense(units=256, activation="softmax")(layer)
    final_2=layers.Dense(units=100, activation="softmax", name="final_layer_2")(layer)

    model=kmodels.Model(inputs=inputlayer, outputs=[final_2, final_0, final_1], name="googlenet")
    return model

model = googlenet()
model.summary()

Model: "googlenet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_46 (Conv2D)  │ (None, 32, 32,    │      9,472 │ input_layer_3[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_14    │ (None, 16, 16,    │          0 │ conv2d_46[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        256 │ max_pooling2d_14… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_47 (Conv2D)  │ (None, 16, 16,    │      4,160 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_48 (Conv2D)  │ (None, 16, 16,    │    110,784 │ conv2d_47[0][0]   │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        768 │ conv2d_48[0][0]   │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_15    │ (None, 8, 8, 192) │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_50 (Conv2D)  │ (None, 8, 8, 96)  │     18,528 │ max_pooling2d_15… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_52 (Conv2D)  │ (None, 8, 8, 16)  │      3,088 │ max_pooling2d_15… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_16    │ (None, 8, 8, 192) │          0 │ max_pooling2d_15… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_49 (Conv2D)  │ (None, 8, 8, 64)  │     12,352 │ max_pooling2d_15… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_51 (Conv2D)  │ (None, 8, 8, 128) │    110,720 │ conv2d_50[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_53 (Conv2D)  │ (None, 8, 8, 32)  │     12,832 │ conv2d_52[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_54 (Conv2D)  │ (None, 8, 8, 32)  │      6,176 │ max_pooling2d_16… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_6       │ (None, 8, 8, 256) │          0 │ conv2d_49[0][0],  │
│ (Concatenate)       │                   │            │ conv2d_51[0][0],  │
│                     │                   │            │ conv2d_53[0][0],  │
│                     │                   │            │ conv2d_54[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_56 (Conv2D)  │ (None, 8, 8, 128) │     32,896 │ concatenate_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_58 (Conv2D)  │ (None, 8, 8, 32)  │      8,224 │ concatenate_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_17    │ (None, 8, 8, 256) │          0 │ concatenate_6[0]

 Total params: 6,513,042 (24.85 MB)

 Trainable params: 6,512,530 (24.84 MB)

 Non-trainable params: 512 (2.00 KB)

In [ ]:
model.compile(optimizer='rmsprop',
              loss='categorical_crossentropy',
              metrics=['accuracy', 'accuracy', 'accuracy'])

In [ ]:
model.fit(x_train, [y1_train, y1_train, y1_train], batch_size=batch_size, epochs=epochs, validation_data=(x_test, [y1_test, y1_test, y1_test]))